In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import os
pathq1 = os.path.join(path, 'Q1_data.csv')
df_q1 = pd.read_csv(pathq1)

In [ ]:
# Task 2: Write your code here:
df_q1.head()

In [ ]:
# Task 3: Write your code here:
df_q1.info()

In [ ]:
# Task 4: Write your code here:
df_q1.describe()

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 5))
plt.hist(df_q1['Delivery_Time'], bins=50, edgecolor='black')
plt.title('Distribution')
plt.show()

In [ ]:
# Task 1: Write your code here:
cols=['Order_ID',	'Distance_km',	'Weather',	'Traffic_Level' ,	'Time_of_Day',	'Vehicle_Type',	'Preparation_Time_min',	'Courier_Experience_yrs',	'Delivery_Time']
df_clean = df_q1[cols].copy()
df_clean= df_clean.drop(columns=['Order_ID'])


In [ ]:
# Task 2: Write your code here:
def check_missing_values(df):
  missing_values = df_q1.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df_clean)

print(f"Before: {df_clean.shape}")
df_clean = df_clean.dropna(subset=['Weather', 'Traffic_Level', 'Time_of_Day', 'Courier_Experience_yrs', 'Delivery_Time'])
print(f"After dropping missing price/year/odometer: {df_clean.shape}")

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import LabelEncoder

categorical_cols = ['Weather',	'Traffic_Level',	'Time_of_Day'	, 'Vehicle_Type']
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

df_clean.head()

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df_clean.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df_clean[numerical_cols] = scaler.fit_transform(df_clean[numerical_cols])
df_clean.head()


In [ ]:
# Task 6: Write your code here:
#it`s reggretion problem so i think imbalance data apllied to classfacation problems

In [ ]:
# Task 1: Write your code here:

# Define features (X) and target (y)
feature_cols = ['Distance_km',	'Weather',	'Traffic_Level' ,	'Time_of_Day',	'Vehicle_Type',	'Preparation_Time_min',	'Courier_Experience_yrs']
X = df_clean[feature_cols]
y = df_clean['Delivery_Time']


In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import  KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

mae_scores = []
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
# Iterate through folds
for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    # print shapes
    print(f"Fold {fold}")
    print("  X_train shape:", X_train.shape)
    print("  X_test shape :", X_test.shape)
    print("  y_train shape:", y_train.shape)
    print("  y_test shape :", y_test.shape)
    print("-" * 30)

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_test, y_pred))



print("Model trained!")
mae_scores = np.array(mae_scores)
print(f"MAE:  ${mae_scores.mean():,.2f}")









In [ ]:
# Task 1: Write your code here:
# Feature importance
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
# Plot for Linear Regression Predictions vs. Ground Truth
plt.figure(figsize=(6, 4))
plt.scatter(y_test, y_pred, alpha=0.7)
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], 'r--', linewidth=2)
plt.xlabel("Actual y_test (Ground Truth)")
plt.ylabel("Predicted y_pred (Linear Regression)")
plt.title("Linear Regression: Predictions vs. Ground Truth")
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Task Bonus: Write your code here: